In [2]:
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET

spark = get_spark("bronze-ingestion")

In [8]:
# Read the sales data from S3

sales_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"s3a://{AWS_BUCKET}/raw/sales/superstore_sales.csv")
)

sales_df.show(5)
sales_df.printSchema()

+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------------------+--------+--------+--------+
|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|Customer Name|    Segment|      Country|        City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|               Sales|Quantity|Discount|  Profit|
+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------------------+--------+--------+--------+
|CA-2019-103800|2019-01-03|2019-01-07|Standard Class|   DP-13000|Darren Powers|   Consumer|United States|     Houston|       Texas|      77095|Central|OFF-PA-10000174|Office Supplies|       Paper|"Messa

In [9]:
# Clean the data by renaming them for consistency

from pyspark.sql.functions import col

sales_df_clean = sales_df.select(
    col("Order ID").alias("order_id"),
    col("Order Date").alias("order_date"),
    col("Ship Date").alias("ship_date"),
    col("Ship Mode").alias("ship_mode"),
    col("Customer ID").alias("customer_id"),
    col("Customer Name").alias("customer_name"),
    col("Segment").alias("segment"),
    col("Country").alias("country"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Postal Code").alias("postal_code"),
    col("Region").alias("region"),
    col("Product ID").alias("product_id"),
    col("Category").alias("category"),
    col("Sub-Category").alias("sub_category"),
    col("Product Name").alias("product_name"),
    col("Sales").alias("sales"),
    col("Quantity").alias("quantity"),
    col("Discount").alias("discount"),
    col("Profit").alias("profit")
)

sales_df_clean.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- profit: double (nullable = true)



In [ ]:
# Write the cleaned data to the Bronze layer in Iceberg format

(
    sales_df_clean.write
    .format("iceberg")
    .mode("overwrite")
    .save("nessie.bronze")
)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view bronze.sales cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.

In [ ]:
# Query the Bronze layer to verify the data is ingested correctly

spark.sql("SELECT * FROM nessie.bronze.sales LIMIT 10").show()

In [ ]:
# Query the Bronze layer to verify the data is ingested correctly

spark.sql("SELECT * FROM nessie.bronze.sales.history").show()